# 🤖 Lab 3 — Full RAG Pipeline

## What We Build Today

```
Medical Book (PDF)
      ↓
① Chunk text (sentence-level — best from Lab 2)
      ↓
② Embed chunks (Ollama nomic-embed-text)
      ↓
③ Store in Weaviate Cloud (Lab 1)
      ↓
④ User Query → Embed → Retrieve top-k
      ↓
⑤ Format → Inject into prompt
      ↓
⑥ Groq LLM → Response
      ↓
⑦ Interactive Widget — compare RAG vs no RAG
```

## 1️⃣ Setup & Connect

In [14]:
import os
import pymupdf as fitz
import nltk
import numpy as np
import weaviate
import weaviate.classes as wvc
import ollama
from groq import Groq
from dotenv import load_dotenv
from nltk.tokenize import sent_tokenize
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
# ── Clean Weaviate — delete all collections ───────────────────────────────────
collections = weaviate_client.collections.list_all()
for name in collections:
    weaviate_client.collections.delete(name)
    print(f'🗑️  Deleted: {name}')

print('✅ Weaviate is clean — ready to ingest')

🗑️  Deleted: MedicalChunks
✅ Weaviate is clean — ready to ingest


In [15]:


nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

load_dotenv()

# ── Clients ───────────────────────────────────────────────────────────────────
weaviate_client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.getenv('WEAVIATE_URL'),
    auth_credentials=weaviate.auth.AuthApiKey(os.getenv('WEAVIATE_API_KEY'))
)
groq_client = Groq(api_key=os.getenv('GROQ_API_KEY'))

COLLECTION_NAME = 'MedicalBook'
LLM_MODEL       = 'llama-3.1-8b-instant'
PDF_PATH        = 'data/medical_book.pdf'

print('✅ Weaviate:', weaviate_client.is_ready())
print('✅ Groq client ready')

✅ Weaviate: True
✅ Groq client ready


## 2️⃣ Embedding Function

In [16]:
def embed(text: str) -> list[float]:
    """
    Embed text using Ollama nomic-embed-text (local).
    Returns a list of floats.
    """
    response = ollama.embeddings(model='nomic-embed-text', prompt=text)
    return response['embedding']

# Quick test
test = embed('test')
print(f'✅ Embedding dim: {len(test)}')

✅ Embedding dim: 768


## 3️⃣ Ingest PDF into Weaviate

We use **sentence-level chunking** (best from Lab 2) and store all chunks with their embeddings.

In [9]:
print(fitz.__file__)

/home/abdelalim/projects/stages/RAG/medical-rag-advanced/.venv/lib/python3.11/site-packages/pymupdf/__init__.py


In [17]:

def extract_and_chunk(PDF_PATH, sentences_per_chunk: int = 4, overlap: int = 1) -> list[dict]:
    """Extract PDF text and split into sentence-level chunks."""
    doc    = fitz.open(PDF_PATH)
    chunks = []
    chunk_id = 0

    for page_num, page in enumerate(doc):
        text      = page.get_text().strip()
        if not text:
            continue
        sentences = sent_tokenize(text)
        i = 0
        while i < len(sentences):
            chunk_text = ' '.join(sentences[i:i + sentences_per_chunk]).strip()
            if chunk_text:
                chunks.append({
                    'chunk_id': chunk_id,
                    'text':     chunk_text,
                    'page':     page_num + 1
                })
                chunk_id += 1
            i += sentences_per_chunk - overlap

    doc.close()
    return chunks


def ingest_to_weaviate(chunks: list[dict]):
    """Create collection and insert all chunks with embeddings."""
    # Delete if exists
    if weaviate_client.collections.exists(COLLECTION_NAME):
        weaviate_client.collections.delete(COLLECTION_NAME)
        print(f'🗑️  Deleted existing: {COLLECTION_NAME}')

    # Create collection
    weaviate_client.collections.create(
        name=COLLECTION_NAME,
        vectorizer_config=wvc.config.Configure.Vectorizer.none(),
        properties=[
            wvc.config.Property(name='text',     data_type=wvc.config.DataType.TEXT),
            wvc.config.Property(name='page',     data_type=wvc.config.DataType.INT),
            wvc.config.Property(name='chunk_id', data_type=wvc.config.DataType.INT),
        ]
    )
    print(f'✅ Collection created: {COLLECTION_NAME}')

    # Batch insert
    collection = weaviate_client.collections.get(COLLECTION_NAME)
    total      = len(chunks)

    with collection.batch.dynamic() as batch:
        for i, chunk in enumerate(chunks):
            vector = embed(chunk['text'])
            batch.add_object(
                properties={
                    'text':     chunk['text'],
                    'page':     chunk['page'],
                    'chunk_id': chunk['chunk_id'],
                },
                vector=vector
            )
            if (i + 1) % 50 == 0 or (i + 1) == total:
                print(f'  📥 Inserted {i + 1}/{total} chunks...')

    print(f'\n🎉 Ingestion complete — {total} chunks in Weaviate')


# Run ingestion
print('📄 Extracting and chunking PDF...')
chunks = extract_and_chunk(PDF_PATH)
print(f'✅ {len(chunks)} chunks created')

print('\n📥 Ingesting into Weaviate (this may take a few minutes)...')
ingest_to_weaviate(chunks)

📄 Extracting and chunking PDF...
✅ 2237 chunks created

📥 Ingesting into Weaviate (this may take a few minutes)...
✅ Collection created: MedicalBook
  📥 Inserted 50/2237 chunks...
  📥 Inserted 100/2237 chunks...
  📥 Inserted 150/2237 chunks...
  📥 Inserted 200/2237 chunks...
  📥 Inserted 250/2237 chunks...
  📥 Inserted 300/2237 chunks...
  📥 Inserted 350/2237 chunks...
  📥 Inserted 400/2237 chunks...
  📥 Inserted 450/2237 chunks...
  📥 Inserted 500/2237 chunks...
  📥 Inserted 550/2237 chunks...
  📥 Inserted 600/2237 chunks...
  📥 Inserted 650/2237 chunks...
  📥 Inserted 700/2237 chunks...
  📥 Inserted 750/2237 chunks...
  📥 Inserted 800/2237 chunks...
  📥 Inserted 850/2237 chunks...
  📥 Inserted 900/2237 chunks...
  📥 Inserted 950/2237 chunks...
  📥 Inserted 1000/2237 chunks...
  📥 Inserted 1050/2237 chunks...
  📥 Inserted 1100/2237 chunks...
  📥 Inserted 1150/2237 chunks...
  📥 Inserted 1200/2237 chunks...
  📥 Inserted 1250/2237 chunks...
  📥 Inserted 1300/2237 chunks...
  📥 Inserted 

## 4️⃣ Retrieve Relevant Chunks

In [18]:
def retrieve(query: str, top_k: int = 5) -> list[dict]:
    """
    Embed query → search Weaviate → return top-k chunks.
    """
    query_vector = embed(query)
    collection   = weaviate_client.collections.get(COLLECTION_NAME)

    results = collection.query.near_vector(
        near_vector=query_vector,
        limit=top_k,
        return_metadata=wvc.query.MetadataQuery(distance=True)
    )

    return [
        {
            'text':     obj.properties['text'],
            'page':     obj.properties['page'],
            'distance': obj.metadata.distance
        }
        for obj in results.objects
    ]

# Test retrieval
test_results = retrieve('What diseases affect the skin?', top_k=3)
print('🔍 Test retrieval:')
for r in test_results:
    print(f'  [dist={r["distance"]:.4f}] Page {r["page"]}: {r["text"][:100]}...')

🔍 Test retrieval:
  [dist=0.3556] Page 199: Il se présente comme une lésion blanche plus ou moins rosée, 
mal vascularisée avec des phlyctènes. ...
  [dist=0.3843] Page 138: 138
UE 7. Infla mmation - Immunopathologie - Poumon - San
Liquide 
mécanique
Liquide inflammatoire
L...
  [dist=0.3906] Page 111: y
Diagnostiquer et connaître les principes du traitement d’une arthrite avec ou sans 
matériel, d’un...


## 5️⃣ Format Documents for Prompt

In [19]:
def format_docs(docs: list[dict]) -> str:
    """
    Format retrieved chunks into a structured string
    ready to be injected into the LLM prompt.
    """
    formatted = []
    for i, doc in enumerate(docs, 1):
        formatted.append(f'Document {i} (Page {doc["page"]}):\n{doc["text"]}')
    return '\n\n'.join(formatted)


DEFAULT_PROMPT = (
    'You are a helpful medical assistant. '
    'Use ONLY the following documents to answer the question accurately.\n\n'
    'Documents:\n{documents}\n\n'
    'Question: {query}\n'
    'Answer:'
)

def build_prompt(query: str, top_k: int = 5, prompt_template: str = DEFAULT_PROMPT) -> str:
    """Retrieve docs and build the full RAG prompt."""
    docs      = retrieve(query, top_k)
    formatted = format_docs(docs)
    return prompt_template.format(query=query, documents=formatted)

# Preview prompt
sample_prompt = build_prompt('What is the role of white blood cells?', top_k=3)
print('📝 Sample RAG prompt (first 500 chars):')
print(sample_prompt[:500] + '...')

📝 Sample RAG prompt (first 500 chars):
You are a helpful medical assistant. Use ONLY the following documents to answer the question accurately.

Documents:
Document 1 (Page 120):
IV. ÉTIOLOGIE
A. Causes post-traumatiques
h
h C’est l’étiologie la plus importante. Elle représente environ 65 % des SDRC de 
type 1.

Document 2 (Page 128):
128
UE 7. Inflammation - Immunopathologie - Poumon - Sang
I. INTRODUCTION
La membrane synoviale tapissant la cavité articulaire sécrète à l’état physiologique du 
liquide synovial en faible quantité, qu...


## 6️⃣ LLM Call via Groq

In [22]:
def llm_call(prompt: str, max_tokens: int = 500) -> str:
    """Send prompt to Groq and return the response content."""
    response = groq_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=max_tokens
    )
    return response.choices[0].message.content


def rag(query: str, use_rag: bool = True, top_k: int = 5) -> str:
    """
    Full RAG pipeline.
    use_rag=True  → retrieve + format + LLM
    use_rag=False → query directly to LLM (no context)
    """
    if use_rag:
        prompt = build_prompt(query, top_k)
    else:
        prompt = query
    return llm_call(prompt)


# Quick test
query = 'difference entre les tumeurs des os primitives et secondaires?'
print('📚 WITH RAG:')
print(rag(query, use_rag=True))
print('\n🤖 WITHOUT RAG:')
print(rag(query, use_rag=False))

📚 WITH RAG:
Selon les documents fournis, voici la différence entre les tumeurs des os primitives et secondaires :

- **Tumeurs des os primitives** :
 + Plus rares (< 0,7/100 000/an)
 + Touchent plus souvent les sujets jeunes
 + Naissent dans l'os et sont le plus souvent uniques
 + Peuvent être bénignes (les plus fréquentes) ou malignes
 + Exemples : chordome, infections primitives à germe banal ou mal de Pott (document 4)

- **Tumeurs des os secondaires (ou métastatiques)** :
 + Plus fréquentes
 + Lien avec la dissémination d'une tumeur cancéreuse développée ailleurs dans l'organisme (sein, prostate, poumon, rein, etc.)
 + Exemples : cancers primitifs, hémopathies malignes comme le myélome multiple
 + Les facteurs de risque des cancers secondaires des os sont identiques à ceux des cancers primitifs (document 5)

Ainsi, la principale différence entre les tumeurs des os primitives et secondaires est leur origine et leur fréquence. Les tumeurs primitives sont plus rares et naissent dans l

## 7️⃣ Interactive Test Widget

Compare **WITH RAG** vs **WITHOUT RAG** side by side.

In [21]:
def display_rag_widget():
    # ── UI Components ─────────────────────────────────────────
    query_input = widgets.Text(
        placeholder='e.g. What diseases affect the skin?',
        description='Query:',
        layout=widgets.Layout(width='100%')
    )
    topk_slider = widgets.IntSlider(
        value=5, min=1, max=15, step=1,
        description='Top K:',
        style={'description_width': 'initial'}
    )
    prompt_input = widgets.Textarea(
        placeholder='Optional: custom prompt template with {query} and {documents}',
        description='Prompt:',
        layout=widgets.Layout(width='100%', height='80px'),
        style={'description_width': 'initial'}
    )
    run_btn    = widgets.Button(
        description='🚀 Run',
        button_style='primary',
        layout=widgets.Layout(width='150px')
    )
    status     = widgets.Output()
    out_rag    = widgets.Output(layout={'border': '2px solid #4CAF50', 'padding': '10px',
                                        'width': '48%', 'min_height': '200px'})
    out_nrag   = widgets.Output(layout={'border': '2px solid #F44336', 'padding': '10px',
                                        'width': '48%', 'min_height': '200px'})

    label_rag  = widgets.HTML('<b style="color:#4CAF50">📚 WITH RAG</b>')
    label_nrag = widgets.HTML('<b style="color:#F44336">🤖 WITHOUT RAG</b>')

    def on_run(b):
        out_rag.clear_output()
        out_nrag.clear_output()
        status.clear_output()

        query    = query_input.value.strip()
        top_k    = topk_slider.value
        template = prompt_input.value.strip() or DEFAULT_PROMPT

        if not query:
            with status:
                print('⚠️  Please enter a query.')
            return

        with status:
            print('⏳ Generating responses...')

        # WITH RAG
        with out_rag:
            prompt = build_prompt(query, top_k, template)
            resp   = llm_call(prompt)
            display(Markdown(resp))

        # WITHOUT RAG
        with out_nrag:
            resp = llm_call(query)
            display(Markdown(resp))

        status.clear_output()

    run_btn.on_click(on_run)

    # ── Layout ────────────────────────────────────────────────
    display(
        widgets.HTML('<h3>🧪 RAG Test Widget</h3>'),
        query_input,
        topk_slider,
        prompt_input,
        run_btn,
        status,
        widgets.HBox([label_rag,  label_nrag],  layout=widgets.Layout(justify_content='space-around')),
        widgets.HBox([out_rag,    out_nrag],    layout=widgets.Layout(justify_content='space-between'))
    )

display_rag_widget()

HTML(value='<h3>🧪 RAG Test Widget</h3>')

Text(value='', description='Query:', layout=Layout(width='100%'), placeholder='e.g. What diseases affect the s…

IntSlider(value=5, description='Top K:', max=15, min=1, style=SliderStyle(description_width='initial'))

Textarea(value='', description='Prompt:', layout=Layout(height='80px', width='100%'), placeholder='Optional: c…

Button(button_style='primary', description='🚀 Run', layout=Layout(width='150px'), style=ButtonStyle())

Output()

## 8️⃣ Close Connection

In [ ]:
weaviate_client.close()
print('✅ Done — Weaviate connection closed')

## ✅ Lab 3 Summary

| Step | What we did |
|---|---|
| **Ingest** | PDF → sentence chunks → embed → Weaviate |
| **Retrieve** | Query → embed → Weaviate similarity search |
| **Format** | Top-k chunks → structured prompt |
| **Generate** | Groq LLM → grounded response |
| **Compare** | Side-by-side RAG vs no RAG widget |

### Full Stack
```
PDF → PyMuPDF
Chunking → Sentence-level (NLTK)
Embeddings → Ollama nomic-embed-text (local)
Vector DB → Weaviate Cloud
LLM → Groq llama-3.1-8b-instant (free)
```